In [ ]:
from pathlib import Path
import sys
import matplotlib as mpl
CAMERA_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'figure_generation_notebooks' else Path.cwd().resolve()
EXPERIMENT_ROOT = Path('/mnt/e/watermark_rev2')
FIGURETYPE1_DIR = CAMERA_ROOT / 'figuretype1'
FIGURETYPE1_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EXPERIMENT_ROOT))
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


In [ ]:
from scipy.stats import norm
MAX_IDX = 50000
USE_ALL_DATA = True
DATA_DIR = EXPERIMENT_ROOT / 'archive' / 'bleu'
MODELS = ['opt-125m', 'gpt2', 'pythia-160m']
DATASETS = ['c4', 'lfqa', 'wikipedia']
METRICS_MAP = {'bleu': 'bleu', 'rouge': 'rouge', 'bertscore': 'bertscore'}
experiment_params = pd.read_csv(EXPERIMENT_ROOT / 'optimal_power_results_combined.csv')

def load_and_aggregate(metric_key, max_idx=MAX_IDX, use_all_data=USE_ALL_DATA):
    all_agg = []
    for model in MODELS:
        for dataset in DATASETS:
            fname = DATA_DIR / f'quality_data_vsNonWM_{metric_key}_{model}_{dataset}.csv'
            if not os.path.exists(fname):
                continue
            df = pd.read_csv(fname)
            if not use_all_data:
                df = df[df['idx'] < max_idx]
            for row in experiment_params.to_numpy():
                alpha, n, gamma_klt, delta_klt = (row[0], row[1], row[2], row[3])
                z_threshold = norm.ppf(1 - alpha)
                for (method, gamma, delta, length), grp in df.groupby(['method', 'gamma', 'delta', 'length']):
                    if length != n:
                        continue
                    if method == 'klt':
                        if not (abs(gamma - gamma_klt) < 1e-05 and abs(delta - delta_klt) < 1e-05):
                            continue
                    total = len(grp)
                    if total == 0:
                        continue
                    tpr = (grp['zscore'] > z_threshold).sum() / total
                    mean_q = grp['quality'].mean()
                    std_q = grp['quality'].std()
                    all_agg.append({'model': model, 'dataset': dataset, 'method': method, 'metric': metric_key, 'alpha': alpha, 'n': int(n), 'gamma': gamma, 'delta': delta, 'TPR': tpr, 'mean_quality': mean_q, 'std_quality': std_q, 'n_samples': total})
    return pd.DataFrame(all_agg)
if USE_ALL_DATA:
    pass
else:
    pass
df_bleu = load_and_aggregate('bleu')
df_rouge = load_and_aggregate('rouge')
df_bert = load_and_aggregate('bertscore')


In [ ]:
def count_plot_points(df, model, dataset, alpha, n, min_tpr=0.8):
    subset = df[(df['model'] == model) & (df['dataset'] == dataset) & (df['alpha'] == alpha) & (df['n'] == n)].copy()
    before_tpr_filter = len(subset)
    subset = subset[subset['TPR'] > min_tpr]
    after_tpr_filter = len(subset)
    method_counts = subset.groupby('method').size().to_dict()
    return {'before_tpr_filter': before_tpr_filter, 'after_tpr_filter': after_tpr_filter, 'by_method': method_counts}
total_points = 0
for ds in ['c4', 'lfqa', 'wikipedia']:
    for model in ['opt-125m', 'gpt2', 'pythia-160m']:
        stats = count_plot_points(df_bleu, model, ds, alpha=0.05, n=50, min_tpr=0.8)
        total_points += stats['after_tpr_filter']
example = df_bleu[(df_bleu['model'] == 'opt-125m') & (df_bleu['dataset'] == 'c4') & (df_bleu['alpha'] == 0.05) & (df_bleu['n'] == 50)].copy()


In [ ]:
UNIFORM_POINTS = True
PARETO_SIZE = 30
PARETO_ALPHA = 0.7
LABEL_FONTSIZE = 12
TICK_FONTSIZE = 12
LEGEND_FONTSIZE = 12
SHOW_TICKS = True
CONFIG = {'xlims_bleu': (0.015, 0.1), 'xlims_rouge': (0.1, 0.5), 'xlims_bert': (0.8, 1.0), 'ylims': (0, 1.02), 'auto_xlim': False, 'auto_ylim': False, 'xlim_padding': 0.05, 'ylim_padding': 0.02}
COLORS = {'grid': '#1f77b4', 'dp': '#2ca02c', 'klt': '#d62728', 'opt': '#9467bd', 'dip': '#ff7f0e'}
LABELS = {'klt': 'Ours', 'dp': 'DP', 'grid': 'KGW', 'opt': 'OPT', 'dip': 'DiPmark'}
METHODS = ['grid', 'dp', 'klt', 'opt', 'dip']
MIN_CURVATURE = 3
MAX_CURVATURE = 1500


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit

def set_global_style(font_scale=1.2):
    sns.set_theme(style='white')
    sns.set_context('paper', font_scale=font_scale)
    plt.rcParams.update({'axes.grid': False, 'grid.alpha': 0.0, 'axes.edgecolor': 'black', 'axes.linewidth': 1.0})

def compute_auto_lims(x, y, cfg, metric='bleu'):
    x = np.asarray(x)
    y = np.asarray(y)
    x_valid = x[np.isfinite(x)]
    y_valid = y[np.isfinite(y)]
    if len(x_valid) == 0 or len(y_valid) == 0:
        defaults = {'bleu': (0, 0.15), 'rouge': (0, 0.5), 'bertscore': (0.8, 1.0)}
        return (cfg.get(f'xlims_{metric}', defaults.get(metric, (0, 1))), cfg.get('ylims', (0.825, 1.02)))
    x_min, x_max = (x_valid.min(), x_valid.max())
    y_min, y_max = (y_valid.min(), y_valid.max())
    x_padding = max((x_max - x_min) * cfg.get('xlim_padding', 0.05), 0.01)
    y_padding = max((y_max - y_min) * cfg.get('ylim_padding', 0.02), 0.01)
    return ((x_min - x_padding, x_max + x_padding), (y_min - y_padding, min(y_max + y_padding, 1.02)))

def style_axes_clean(ax):
    ax.set_facecolor('white')
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for side in ['left', 'bottom']:
        ax.spines[side].set_color('black')
        ax.spines[side].set_linewidth(1.0)
    if SHOW_TICKS:
        ax.tick_params(axis='both', colors='black', direction='out', length=3, width=1, labelsize=TICK_FONTSIZE)
    else:
        ax.tick_params(axis='both', which='both', length=0, labelbottom=False, labelleft=False)

def is_pareto_front(x, y, eps_x=0.0, eps_y=0.0):
    n = len(x)
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if x[j] >= x[i] + eps_x and y[j] >= y[i] + eps_y:
                keep[i] = False
                break
    return keep

def adaptive_pareto(x, y, min_points=5, eps_step_x=0.001, eps_step_y=0.001, max_iter=200):
    if len(x) <= min_points:
        return (np.ones(len(x), dtype=bool), 0.0, 0.0)
    eps_x = 0.0
    eps_y = 0.0
    for it in range(max_iter):
        mask = is_pareto_front(x, y, eps_x, eps_y)
        if np.sum(mask) >= min_points:
            return (mask, eps_x, eps_y)
        if it % 2 == 0:
            eps_x += eps_step_x
        else:
            eps_y += eps_step_y
    mask = is_pareto_front(x, y, eps_x, eps_y)
    return (mask, eps_x, eps_y)

def inverse_sigmoid(x, a, b):
    return 1.0 / (1.0 + np.exp(a * (x - b)))

def fit_inverse_sigmoid(x, y, min_curvature=None, max_curvature=None, anchor_left=(0, 1), anchor_right=(3, 0)):
    if min_curvature is None:
        min_curvature = MIN_CURVATURE
    if max_curvature is None:
        max_curvature = MAX_CURVATURE
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = (x[valid], y[valid])
    if len(x) < 2:
        return None
    real_x_min, real_x_max = (float(x.min()), float(x.max()))
    anchor_x = np.array([anchor_left[0], anchor_right[0]])
    anchor_y = np.array([anchor_left[1], anchor_right[1]])
    x_fit = np.concatenate([x, anchor_x])
    y_fit = np.concatenate([y, anchor_y])
    eps = 0.0001
    y_clip = np.clip(y_fit, eps, 1 - eps)
    logit = np.log(y_clip / (1 - y_clip))
    m_coef, c_coef = np.polyfit(x_fit, logit, 1)
    a0 = -m_coef
    b0 = c_coef / a0 if a0 != 0 else np.median(x_fit)
    a0 = np.clip(abs(a0), min_curvature, max_curvature)
    try:
        popt, _ = curve_fit(inverse_sigmoid, x_fit, y_clip, p0=[a0, b0], bounds=([min_curvature, x_fit.min() - 10], [max_curvature, x_fit.max() + 10]), maxfev=20000)
        a, b = popt
    except Exception:
        a, b = (a0, b0)
    y_pred = inverse_sigmoid(x, a, b)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {'a': float(a), 'b': float(b), 'r2': float(r2), 'n_points': len(x), 'x_min': real_x_min, 'x_max': real_x_max}

def plot_panel(ax, subset, cfg, metric='bleu', point_alpha=0.7, point_size=30, non_pareto_alpha=0.15, show_legend=False, show_ylabel=True, auto_xlim=True, auto_ylim=True, min_tpr=0.8, show_fit_curve=True, curve_alpha=0.6, curve_extend_left=0.01, curve_extend_right=0.01, min_pareto_points=5, pareto_eps_step_x=0.001, pareto_eps_step_y=0.001, pareto_max_iter=200, anchor_left=(-5, 1), anchor_right=(5, 0)):
    if subset is None or len(subset) == 0:
        ax.set_visible(False)
        return
    subset = subset[subset['TPR'] > min_tpr].copy()
    if len(subset) == 0:
        ax.set_visible(False)
        return
    y = subset['TPR'].values
    x = subset['mean_quality'].values
    methods_col = subset['method'].values
    auto_xlims, auto_ylims = compute_auto_lims(x, y, cfg, metric)
    xlims = auto_xlims if auto_xlim else cfg.get(f'xlims_{metric}', (0, 1))
    ylims = auto_ylims if auto_ylim else cfg.get('ylims', (0.825, 1.02))
    if UNIFORM_POINTS:
        actual_pareto_size = PARETO_SIZE
        actual_pareto_alpha = PARETO_ALPHA
        actual_np_size = PARETO_SIZE
        actual_np_alpha = PARETO_ALPHA
    else:
        actual_pareto_size = point_size
        actual_pareto_alpha = point_alpha
        actual_np_size = point_size * 0.6
        actual_np_alpha = non_pareto_alpha
    style_axes_clean(ax)
    for m in METHODS:
        m_mask = methods_col == m
        if not np.any(m_mask):
            continue
        mx = x[m_mask]
        my = y[m_mask]
        pareto_mask, used_eps_x, used_eps_y = adaptive_pareto(mx, my, min_points=min_pareto_points, eps_step_x=pareto_eps_step_x, eps_step_y=pareto_eps_step_y, max_iter=pareto_max_iter)
        if np.any(~pareto_mask):
            sns.scatterplot(x=mx[~pareto_mask], y=my[~pareto_mask], ax=ax, color=COLORS[m], s=actual_np_size, alpha=actual_np_alpha, edgecolor='white', linewidth=0.3, legend=False)
        if np.any(pareto_mask):
            sns.scatterplot(x=mx[pareto_mask], y=my[pareto_mask], ax=ax, color=COLORS[m], s=actual_pareto_size, alpha=actual_pareto_alpha, edgecolor='white', linewidth=0.4, label=LABELS[m], legend=False)
        else:
            ax.scatter([], [], label=LABELS[m], color=COLORS[m], marker='o', s=actual_pareto_size)
        n_pareto = np.sum(pareto_mask)
        if show_fit_curve and n_pareto >= 2:
            fit_res = fit_inverse_sigmoid(mx[pareto_mask], my[pareto_mask], anchor_left=anchor_left, anchor_right=anchor_right)
            if fit_res is not None:
                x_left = fit_res['x_min'] - curve_extend_left
                x_right = fit_res['x_max'] + curve_extend_right
                x_curve = np.linspace(x_left, x_right, 200)
                y_curve = inverse_sigmoid(x_curve, fit_res['a'], fit_res['b'])
                ax.plot(x_curve, y_curve, color=COLORS[m], linewidth=2, alpha=curve_alpha)
    ax.set_xlabel(f'{metric.upper()}', fontsize=LABEL_FONTSIZE)
    ax.set_ylabel('TPR' if show_ylabel else '', fontsize=LABEL_FONTSIZE)
    ax.set_xlim(xlims)
    ax.set_ylim(ylims)
    if show_legend:
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            legend = ax.legend(loc='lower left', fontsize=LEGEND_FONTSIZE, frameon=True)
            frame = legend.get_frame()
            frame.set_facecolor('white')
            frame.set_edgecolor('#cccccc')

def plot_three_metrics_vertical(df_bleu, df_rouge, df_bert, model, dataset, alpha, n, cfg=None, font_scale=1.2, point_alpha=0.7, point_size=30, non_pareto_alpha=0.15, auto_xlim=True, auto_ylim=True, min_tpr=0.8, show_fit_curve=True, curve_alpha=0.6, curve_extend_left=0.01, curve_extend_right=0.01, min_pareto_points=5, pareto_eps_step_x=0.001, pareto_eps_step_y=0.001, pareto_max_iter=200, anchor_left=(-5, 1), anchor_right=(5, 0), save=True, out_dir=str(FIGURETYPE1_DIR)):
    set_global_style(font_scale)
    cfg = cfg or CONFIG

    def filter_data(df):
        return df[(df['model'] == model) & (df['dataset'] == dataset) & (df['alpha'] == alpha) & (df['n'] == n)].copy()
    metrics_data = [('bleu', filter_data(df_bleu), 'BLEU'), ('rouge', filter_data(df_rouge), 'ROUGE'), ('bertscore', filter_data(df_bert), 'BERTScore')]
    available = [(m, d, l) for m, d, l in metrics_data if len(d) > 0]
    if not available:
        return
    n_rows = len(available)
    FIG_WIDTH = 6
    PANEL_HEIGHT = 2
    fig, axes = plt.subplots(n_rows, 1, figsize=(FIG_WIDTH, PANEL_HEIGHT * n_rows), facecolor='white')
    axes_list = [axes] if n_rows == 1 else list(axes)
    middle_idx = n_rows // 2
    for i, (metric, subset, label) in enumerate(available):
        plot_panel(axes_list[i], subset, cfg, metric, point_alpha=point_alpha, point_size=point_size, non_pareto_alpha=non_pareto_alpha, show_legend=i == n_rows - 1, show_ylabel=i == middle_idx, auto_xlim=auto_xlim, auto_ylim=auto_ylim, min_tpr=min_tpr, show_fit_curve=show_fit_curve, curve_alpha=curve_alpha, curve_extend_left=curve_extend_left, curve_extend_right=curve_extend_right, min_pareto_points=min_pareto_points, pareto_eps_step_x=pareto_eps_step_x, pareto_eps_step_y=pareto_eps_step_y, pareto_max_iter=pareto_max_iter, anchor_left=anchor_left, anchor_right=anchor_right)
    plt.tight_layout()
    if save:
        os.makedirs(out_dir, exist_ok=True)
        fname = f'{out_dir}/quality_3metrics_vertical_{model}_{dataset}_a{alpha}_n{int(n)}.pdf'
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


In [ ]:
import warnings
warnings.filterwarnings('ignore')
for ds in ['c4', 'lfqa', 'wikipedia']:
    for model in ['opt-125m', 'gpt2', 'pythia-160m']:
        plot_three_metrics_vertical(df_bleu, df_rouge, df_bert, model=model, dataset=ds, alpha=0.05, n=50, point_alpha=0.7, non_pareto_alpha=0.7, min_pareto_points=3, pareto_eps_step_x=0.001, pareto_eps_step_y=0.001, pareto_max_iter=200)
